## PROJET PARIS TRAFFIC Part 2


**Contexte** : Cette partie va nous permettre de nettoyer les données et de justifier chaque choix avec tes chiffres. Tout d'abord on va telecharger les données.

**Telechargement des données**

In [1]:
import os 
import pandas as pd
import matplotlib.pyplot as plt
CHEMIN = "data/raw/comptages.csv"

**Le chargement par chunks**

In [2]:
TYPES = {
    "iu_ac": "string", "iu_nd_amont": "string", "iu_nd_aval": "string",
    "libelle": "category", "etat_trafic": "category", "etat_barre": "category",
}

morceaux = []
for chunk in pd.read_csv(CHEMIN, sep=";", encoding="utf-8-sig",
                         dtype=TYPES, chunksize=50_000):
    morceaux.append(chunk)

df = pd.concat(morceaux, ignore_index=True)
df.shape

(95960, 11)

## 1. Dates en datetime et en heure de Paris, identifiants en texte, états en category

*a/ Dates en datetime et en heure de Paris* 

Regardons d'abord à quoi ressemble t_1h :

In [3]:
print(df["t_1h"].dtype, df["t_1h"].iloc[0])
df["t_1h"].head(3)


str 2026-08-31T23:00:00+00:00


0    2026-08-31T23:00:00+00:00
1    2026-08-31T23:00:00+00:00
2    2026-08-31T23:00:00+00:00
Name: t_1h, dtype: str

In [4]:
df["t_1h"].str.len().value_counts()
df["t_1h"].str[-6:].value_counts()

t_1h
+00:00    95960
Name: count, dtype: int64

t_1h est du texte (format str) , toutes les lignes finissent par +00:00 sans exception donc on a pas de format mélanger , ils ont le format AAAA-MM-JJTHH:MM:SS+00:00.

Puis la conversion

In [5]:
df["t_utc"] = pd.to_datetime(df["t_1h"], utc=True)
df["t_paris"] = df["t_utc"].dt.tz_convert("Europe/Paris")
df[["t_1h", "t_utc", "t_paris"]].head(3)

,t_1h,t_utc,t_paris
0,2026-08-31T23:00:00+00:00,2026-08-31 23:00:00+00:00,2026-09-01 01:00:00+02:00
1,2026-08-31T23:00:00+00:00,2026-08-31 23:00:00+00:00,2026-09-01 01:00:00+02:00
2,2026-08-31T23:00:00+00:00,2026-08-31 23:00:00+00:00,2026-09-01 01:00:00+02:00


In [6]:
print(df["t_utc"].dtype)
print(df["t_paris"].dtype)
print("Dates non converties :", df["t_utc"].isna().sum())

datetime64[us, UTC]
datetime64[us, Europe/Paris]
Dates non converties : 0


Vérifions que mes données couvrent bien la période du 1er mars au 31 août sans trou au début ni à la fin. Je regarde aussi l'effet de la conversion sur les bords de la période.

In [7]:
print("UTC   :", df["t_utc"].min(), "->", df["t_utc"].max())
print("Paris :", df["t_paris"].min(), "->", df["t_paris"].max())

UTC   : 2026-03-01 00:00:00+00:00 -> 2026-08-31 23:00:00+00:00
Paris : 2026-03-01 01:00:00+01:00 -> 2026-09-01 01:00:00+02:00


En UTC, du 01/03 00h au 31/08 23h : le fichier couvre exactement la période.
Pour Paris du 01/03 01h (+01:00) au 01/09 01h (+02:00) : la fenêtre est décalée de 1 h au début et de 2 h à la fin. Le passage de +01:00 à +02:00 montre aussi que le changement d'heure est pris en compte.

L'étiquette `+00:00` affirme que les heures sont en UTC, mais une étiquette peut être fausse. Si je convertis sans vérifier, toutes mes heures seraient décalées d'1 ou 2 h, et mes heures de pointe seraient fausses. Je teste donc avec un jour où l'heure change : le 29 mars 2026, Paris passe de 02h à 03h, donc cette journée ne dure que 23 h en heure locale.

In [8]:
un_capteur = df[df["iu_ac"] == df["iu_ac"].iloc[0]]

for jour in ["2026-03-28", "2026-03-29", "2026-03-30"]:
    n_utc = (un_capteur["t_utc"].dt.strftime("%Y-%m-%d") == jour).sum()
    n_paris = (un_capteur["t_paris"].dt.strftime("%Y-%m-%d") == jour).sum()
    print(jour, "| UTC :", n_utc, "heures | Paris :", n_paris, "heures")

2026-03-28 | UTC : 24 heures | Paris : 24 heures
2026-03-29 | UTC : 24 heures | Paris : 23 heures
2026-03-30 | UTC : 24 heures | Paris : 24 heures


On a UTC : 24/ 24/24 donc la série horaire n'a aucune heure manquante, y compris le 29 mars. Puis pour Paris 24/23/24 la journée du 29 mars compte 23 heures en heure locale donc que tz_convert gère le changement d'heure. Tout est ok!

Verifions maintenant sur toute la période que le décalage passe bien de +1 h (hiver) à +2 h (été), et pas seulement le 29 mars.

In [9]:
df["decalage"] = df["t_paris"].dt.strftime("%z")
df.groupby(df["t_paris"].dt.month)["decalage"].agg(lambda s: s.unique().tolist())

t_paris
3    [+0200, +0100]
4           [+0200]
5           [+0200]
6           [+0200]
7           [+0200]
8           [+0200]
9           [+0200]
Name: decalage, dtype: object

Le passage de l'heure d'été à bien été appliqué sur toute la période comme on voulait!

In [10]:
df = df.drop(columns="decalage") #pour ne pas encombrer le dataframe avec une colonne temporaire

In [ ]:
df = df.drop(columns="t_1h") #inutile maintenant que l'on a t_utc et t_paris

*b/ Identifiants en texte et états en `category`*


In [12]:
df.dtypes

iu_ac                                     string
libelle                                 category
q                                        float64
k                                        float64
etat_trafic                             category
iu_nd_amont                               string
libelle_nd_amont                             str
iu_nd_aval                                string
libelle_nd_aval                              str
etat_barre                              category
t_utc                        datetime64[us, UTC]
t_paris             datetime64[us, Europe/Paris]
dtype: object

On voit que on a les bon type.